# 03 - Baseline Models (E1-E4)

Trains and compares: E1 majority reference, E2 TF-IDF + Logistic Regression, E3 TF-IDF + classical ML, E4 credibility-only (Section 10). E2's result is the number every later model must beat before being described as an improvement (Section 18).

In [ ]:
import pandas as pd
from ml.preprocessing.clean_text import minimal_normalise
from ml.preprocessing.dataset_split import TARGET_COLUMN

df = pd.read_csv('../ml/data/processed/emscad_processed_split.csv')
text_cols = ['title', 'company_profile', 'description', 'requirements', 'benefits']
df['combined_text'] = df[text_cols].fillna('').agg(' '.join, axis=1).apply(minimal_normalise)

train_df = df[df['split'] == 'train']
val_df = df[df['split'] == 'validation']
print(len(train_df), len(val_df))

## E1 - Majority reference

In [ ]:
from sklearn.dummy import DummyClassifier
from ml.evaluation.metrics import compute_metrics

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(train_df[['combined_text']], train_df[TARGET_COLUMN])
dummy_probs = dummy.predict_proba(val_df[['combined_text']])[:, 1]
compute_metrics(val_df[TARGET_COLUMN], dummy_probs).to_dict()

## E2 - TF-IDF + Logistic Regression

In [ ]:
from ml.models.baseline_tfidf import TfidfLogRegBaseline

e2 = TfidfLogRegBaseline()
e2.fit(train_df['combined_text'], train_df[TARGET_COLUMN])
e2_probs = e2.predict_proba(val_df['combined_text'])
compute_metrics(val_df[TARGET_COLUMN], e2_probs).to_dict()

## E3 - Classical ML (Random Forest on TF-IDF)

In [ ]:
from ml.models.baseline_ml import ClassicalMlBaseline, ClassicalMlConfig

e3 = ClassicalMlBaseline(ClassicalMlConfig(classifier='random_forest'))
e3.fit(train_df['combined_text'], train_df[TARGET_COLUMN])
e3_probs = e3.predict_proba(val_df['combined_text'])
compute_metrics(val_df[TARGET_COLUMN], e3_probs).to_dict()

## E4 - Credibility-only model

In [ ]:
from ml.credibility.credibility_model import CredibilityOnlyModel

e4 = CredibilityOnlyModel()
e4.fit(train_df, train_df[TARGET_COLUMN])
e4_probs = e4.predict_proba(val_df)
compute_metrics(val_df[TARGET_COLUMN], e4_probs).to_dict()

## Save E2 as the deployed baseline

This mirrors what `python -m ml.train_baseline` does automatically; run that script for the actual deployment artefact.